# Fischers Test

Created by Lilly Shatford-Adams on 9/4/2026

The purpose of this script is to answer the biological question of, "Are there sex differences in the prevalance of any NDDs, viruses, or autoimmune diseases?".

We'll answer this question using the Fischer's Tests of F/M vs True(case)/False(control) of NDD/virus/autoimmune disease. Controls will be the absence of NDD/virus/autoimmune disease and Cases will the presence of the NDD/virus/autoimmune disease.


**Fisher's Exact Test**

- Determines if there is a significant association between two categorical varaibles by analyzing a contingency table (typically, or in this case, 2x2 table AKA F/M vs Control/Case). - This calculates the p-value based on hypergeometric distribution. 
- We are looking for *non-random* associations between two variables. 
- It is non-parametric and returns an exact p-value.

It will output a *odds ratio* and *p-value*.

The odds ratio will measure the effect size/strength of the association between every situation (ex. Female AND control, Male AND control). This is the ratio of the odds that an event occurring in one group to the group of it occurring in another group. It provides **magnitude** and **direction**.

- OR > 1 = higher in first group
- OR < 1 = lower in first group (ex. OR 0.5 means even is half as likely to occur in first group compared to second group)

The p-value will measure whether the number is statistically significant. This is the probability that the result is **extreme** in one situation vs the other (null hypothesis = no association == TRUE).

- p < 0.05 = statistically significant
- p > 0.05 = stastistically insignificant

**Contingency Tables**

Will be set up as such:



In [1]:
# load libraries

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

# Fisher Test for all NDDs

In [ ]:
#CONFIGURATION: Select NDDs
ndd_list = ['DEM', 'AD', 'PD', 'VAS']
print(ndd_list)

In [ ]:
!pwd

In [ ]:
results_ndd = []

for ndd in ndd_list:
    
    #configure
    ndd = ndd
    date='SEPTEMBER_02_2025'
    # df = pd.read_csv(f'{ndd}_with_tenure_{date}.csv', low_memory = False)
    df = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/final_files/notebooks/from_siri_rerun/data/{ndd}_JULY_23_2026_ready_cox.csv',low_memory=False)
    
    #prepare contingency table
    contingency_table = pd.crosstab(df["SEX"], df[f"{ndd}_DATE"].notna())
    contingency_table = contingency_table.reindex(columns=[True, False])
    print(contingency_table)
    
    # actual test
    odds,pvalue = fisher_exact(contingency_table, alternative='two-sided')
    print(f'for {ndd}, the Odds Ratio is: ',odds,'and the p-value: ',pvalue)
    
    # append the results into a dictionary
    results_ndd.append({
        "NDD": ndd,
        "OddsRatio": odds,
        "Pvalue": pvalue
        })

In [ ]:
# make the results into a table and add a 'SIG' column (boolean)
results_ndd = pd.DataFrame(results_ndd)
results_ndd["SIG"] = results_ndd["Pvalue"] < 0.05
results_ndd["MALES_HIGHER"] = results_ndd["OddsRatio"] > 1.0
results_ndd['PERCENT'] = (results_ndd['OddsRatio']-1)*100
results_ndd

# Fisher Test for Autoimmune and Virus

- will use a "large" df comprised from dem, ad, pd, and vascular_dem; see OLD CODE for NDD-specific datasets

In [11]:
# read dfs
ad = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/final_files/notebooks/from_siri_rerun/data/AD_JULY_23_2026_ready_cox.csv', low_memory=False)
pa = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/final_files/notebooks/from_siri_rerun/data/PD_JULY_23_2026_ready_cox.csv', low_memory=False)
dem = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/final_files/notebooks/from_siri_rerun/data/DEM_JULY_23_2026_ready_cox.csv', low_memory=False)
vascular_dem = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/final_files/notebooks/from_siri_rerun/data/VAS_JULY_23_2026_ready_cox.csv', low_memory=False)

In [ ]:
# big intersection of the dem and ad datasets

full = pd.merge(dem, ad[['ID','AD_DATE']], how='left', left_on='ID', right_on='ID')
#full.columns
full2 = pd.merge(full, pa[['ID','PD_DATE']], how='left', left_on='ID', right_on='ID')
full3 = pd.merge(full2, vascular_dem[['ID','VAS_DATE']], how='left', left_on='ID', right_on='ID')
full3.shape # should be 86

In [14]:
# reorder the columns for ease..
cols_of_interest = ['ID','DEM_DATE', 'AD_DATE', 'VAS_DATE','PD_DATE']
cols = cols_of_interest + [c for c in full.columns if not c in cols_of_interest]
full_organized = full3[cols]

In [ ]:
full_organized

In [ ]:
full_organized.QC0_L12_VITILIGO.value_counts()

## Fishers for Viruses (31 models)

In [ ]:
codes = pd.read_csv('../../data/labels.csv')
codes = codes[['FinnGen_Phenocode','ICD10_Codes','Cohort','Type','UKB_Description_For_Plots']]
codes = codes[codes['Cohort']=='UKB']
codes = codes[codes['Type']=='virus']
virus_list = ['QC0_' + code for code in codes['FinnGen_Phenocode'].tolist()]
virus_list = [
    virus for virus in virus_list
    if virus in full_organized.columns
]
print(virus_list)
print(len(virus_list))

In [ ]:
results_virus_full = []

for virus in virus_list:
    virus = virus
    df = full_organized.copy()
    number_of_individuals = len(df)
    #prepare contingency table
    contingency_table = pd.crosstab(df["SEX"], df[f'{virus}'])
    contingency_table = contingency_table.reindex(columns=[1, 0]) # flip for directionality - 1 = True (has virus), 0 = False (Virus not present)
    # print(contingency_table)

    #actual test
    odds,pvalue = fisher_exact(contingency_table, alternative='two-sided')
    print(f'within the LARGE DATASET THAT has {number_of_individuals} individuals looking at: ', {virus}, 'virus - the Odds Ratio is: ',odds,'and the p-value: ',pvalue)
        
    # append the results into a dictionary
    results_virus_full.append({
            "Virus": virus,
            "OddsRatio": odds,
            "Pvalue": pvalue
        })

In [ ]:
# make the results into a table and add a 'SIG' column (boolean)
results_virus_full = pd.DataFrame(results_virus_full)
results_virus_full["SIG"] = results_virus_full["Pvalue"] < 0.05
results_virus_full["MALES_HIGHER"] = results_virus_full["OddsRatio"] > 1.0
results_virus_full['PERCENT'] = (results_virus_full['OddsRatio']-1)*100
results_virus_full

## Fishers for Autoimmune Diseases (29 models)

In [ ]:
# select the codes (AUTOIMMUNE) of interest
codes = pd.read_csv('../../data/labels.csv')
codes = codes[['FinnGen_Phenocode','ICD10_Codes','Cohort','Type','UKB_Description_For_Plots']]
codes = codes[codes['Cohort']=='UKB']
codes = codes[codes['Type']=='autoimmune']
autoimmune_list = ['QC0_' + code for code in codes['FinnGen_Phenocode'].tolist()]
autoimmune_list = [
    autoimmune for autoimmune in autoimmune_list
    if autoimmune in full_organized.columns
]
print(autoimmune_list)
print(len(autoimmune_list))

In [ ]:
results_autoimmune_full = []

for autoimmune in autoimmune_list:
    autoimmune = autoimmune
    df = full_organized.copy()
    number_of_individuals = len(df)
    #prepare contingency table
    contingency_table = pd.crosstab(df["SEX"], df[f'{autoimmune}'])
    contingency_table = contingency_table.reindex(columns=[1, 0]) # flip for directionality
    #print(contingency_table)

    #actual test
    odds,pvalue = fisher_exact(contingency_table, alternative='two-sided')
    print(f'within the LARGE DATASET THAT has {number_of_individuals} individuals looking at: ', {autoimmune}, 'autoimmune - the Odds Ratio is: ',odds,'and the p-value: ',pvalue)
        
    # append the results into a dictionary
    results_autoimmune_full.append({
            "autoimmune": autoimmune,
            "OddsRatio": odds,
            "Pvalue": pvalue
        })

In [ ]:
# make the results into a table and add a 'SIG' column (boolean)
results_autoimmune_full = pd.DataFrame(results_autoimmune_full)
results_autoimmune_full["SIG"] = results_autoimmune_full["Pvalue"] < 0.05
results_autoimmune_full["MALES_HIGHER"] = results_autoimmune_full["OddsRatio"] > 1.0
results_autoimmune_full['PERCENT'] = (results_autoimmune_full['OddsRatio']-1)*100
results_autoimmune_full

# save results

In [61]:
results_ndd.to_csv('data/Q1_fishers_ndd_AUG_20_2026.csv',index=False)
results_virus_full.to_csv('data/Q1_fishers_virus_AUG_20_2026.csv',index=False)
results_autoimmune_full.to_csv('data/Q1_fishers_autoimmune_AUG_20_2026.csv',index=False)